# Transformer Encoder from Scratch (Colab)

This notebook mirrors the sections from `transformer.qmd` and builds a **Transformer encoder** step by step.

Our goal is to understand the mechanics and track the exact shapes of tensors at every single stage, **not** to train a model. We use text tokens to match the tutorial narrative, but the tensor mathematics are exactly identical for Vision Transformers and Diffusion Transformers.


In [ ]:
# Colab setup (safe to run multiple times)
# If running locally, ensure you have these installed.
try:
    import plotly
    import rich
except ImportError:
    !pip install -q torch pandas plotly ipywidgets rich


In [ ]:
import math
import torch
import torch.nn as nn
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, IntSlider, Dropdown

from rich import print
from rich.console import Console
from rich.table import Table

console = Console()

torch.manual_seed(42)

if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"



## 1. Tokenization

Before a transformer can process text, the raw string must be converted into a sequence of tokens. Each token is mapped to a unique integer ID from our vocabulary.

*We will use a simple word-level tokenizer for this demonstration.*


In [ ]:
sentence = "The animal didn't cross the street because it was too tired"
tokens = sentence.split()
vocab = {tok: i for i, tok in enumerate(sorted(set(tokens)))}

# Convert text to tensor of integer IDs
token_ids = torch.tensor([vocab[t] for t in tokens], dtype=torch.long)

print(f"[bold magenta]Tokens:[/bold magenta] {tokens}")
print(f"[bold magenta]Sequence length (N):[/bold magenta] [bold cyan]{len(tokens)}[/bold cyan]")
print(f"[bold magenta]Token IDs tensor shape:[/bold magenta] [bold cyan]{tuple(token_ids.shape)}[/bold cyan]\n")

# Use Rich to display the table of tokens and IDs
table = Table(title="Tokenization Dictionary", title_style="bold magenta")
table.add_column("Position", justify="right", style="cyan", no_wrap=True)
table.add_column("Token", style="green")
table.add_column("Token ID", justify="right", style="yellow")

for i, (tok, tid) in enumerate(zip(tokens, token_ids.tolist())):
    table.add_row(str(i), tok, str(tid))

console.print(table)


## 2. Embedding Layer

An integer ID by itself carries no semantic meaning. The **embedding layer** maps each token ID to a learnable dense vector of dimension $d_{	ext{model}}$.

Notice how the shape changes from `[SequenceLength]` to `[SequenceLength, D_model]`.


In [ ]:
d_model = 32

# 1. Create the embedding layer
embedding_layer = nn.Embedding(num_embeddings=len(vocab), embedding_dim=d_model)

# 2. Pass our token IDs through the embedding layer
X_embed = embedding_layer(token_ids)

print(f"Input [bold yellow]token_ids[/bold yellow] shape: [bold cyan]{tuple(token_ids.shape)}[/bold cyan]  --> [N]")
print(f"Output [bold green]X_embed[/bold green] shape:  [bold cyan]{tuple(X_embed.shape)}[/bold cyan]  --> [N, D_model]\n")

# Let's project these 32-dimensional vectors down to 2D using PCA to visualize them
X_centered = X_embed - X_embed.mean(dim=0, keepdim=True)
U, S, V = torch.pca_lowrank(X_centered, q=2)
X_2d = X_centered @ V[:, :2]

emb_df = pd.DataFrame({
    "token": tokens,
    "pc1": X_2d[:, 0].detach().numpy(),
    "pc2": X_2d[:, 1].detach().numpy(),
    "position": list(range(len(tokens))),
})

fig = px.scatter(
    emb_df, x="pc1", y="pc2", text="token", color="position",
    title="Token Embeddings projected to 2D Space"
)
fig.update_traces(textposition="top center", marker=dict(size=10))


## 3. Positional Encoding

Self-attention treats its input as an unordered set. We need to explicitly inject positional information by adding a **Positional Encoding** (PE) tensor of the exact same shape as our embeddings `[N, D_model]`. We add them element-wise: $X = X_{	ext{embed}} + PE$.


In [ ]:
def sinusoidal_positional_encoding(seq_len: int, d_model: int) -> torch.Tensor:
    pe = torch.zeros(seq_len, d_model)
    position = torch.arange(0, seq_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

# 1. Generate PE for our sequence length
PE = sinusoidal_positional_encoding(seq_len=len(tokens), d_model=d_model)

print(f"[bold green]X_embed[/bold green] shape: [bold cyan]{tuple(X_embed.shape)}[/bold cyan]  --> [N, D_model]")
print(f"[bold blue]PE[/bold blue] shape:      [bold cyan]{tuple(PE.shape)}[/bold cyan]  --> [N, D_model]")

# 2. Add Positional Encoding element-wise
X = X_embed + PE
print(f"Combined [bold magenta]X[/bold magenta] shape: [bold cyan]{tuple(X.shape)}[/bold cyan]  --> [N, D_model]\n")

def plot_pe_dims(max_dim: int = 8):
    max_dim = min(max_dim, d_model)
    pe_df = pd.DataFrame({"position": list(range(len(tokens)))})
    for i in range(max_dim):
        pe_df[f"dim_{i}"] = PE[:, i].numpy()

    fig = go.Figure()
    for i in range(max_dim):
        fig.add_trace(go.Scatter(x=pe_df["position"], y=pe_df[f"dim_{i}"], mode="lines+markers", name=f"dim_{i}"))
    fig.update_layout(title=f"Sinusoidal positional encoding waves (first {max_dim} dims)", xaxis_title="Position", yaxis_title="Value")
    fig.show()



## 4. Self-Attention Step-by-Step

Now we take our position-aware tensor `X` and perform the core self-attention operations explicitly. We will define 3 linear projections to create **Queries (Q)**, **Keys (K)**, and **Values (V)**.

We'll compute the attention scores manually:
$$ 	ext{Attention}(Q, K, V) = 	ext{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V $$


In [ ]:
# 1. Create Linear Projections (W_q, W_k, W_v)
q_proj = nn.Linear(d_model, d_model)
k_proj = nn.Linear(d_model, d_model)
v_proj = nn.Linear(d_model, d_model)

# 2. Project Input X into Q, K, and V
Q = q_proj(X)
K = k_proj(X)
V = v_proj(X)

# Use Rich table for shape tracking
table = Table(title="Q, K, V Projections", title_style="bold magenta")
table.add_column("Tensor", style="white", justify="left")
table.add_column("Shape", style="cyan", justify="right")
table.add_row("Input X", str(tuple(X.shape)))
table.add_row("Query Q", str(tuple(Q.shape)))
table.add_row("Key K", str(tuple(K.shape)))
table.add_row("Value V", str(tuple(V.shape)))
console.print(table)

# 3. Compute Attention Scores (Dot product of Q and K)
# We transpose K's last two dimensions to allow matrix multiplication: [N, d_model] @ [d_model, N] -> [N, N]
scores = Q @ K.transpose(-2, -1)
print(f"[bold yellow]Scores[/bold yellow] (Q @ K^T) shape: [bold cyan]{tuple(scores.shape)}[/bold cyan]  --> [N, N]")

# 4. Scale and Softmax
scale_factor = math.sqrt(d_model)
scaled_scores = scores / scale_factor
weights = torch.softmax(scaled_scores, dim=-1)
print(f"[bold yellow]Attention Weights[/bold yellow] shape: [bold cyan]{tuple(weights.shape)}[/bold cyan]  --> [N, N]")
print(f"Proof that row 0 sums to 1.0 (Probability Distribution): [bold green]{weights[0].sum().item():.4f}[/bold green]\n")

# 5. Compute Weighted Sum of Values
attention_output = weights @ V
print(f"[bold magenta]Attention Output[/bold magenta] shape (Weights @ V): [bold cyan]{tuple(attention_output.shape)}[/bold cyan]  --> [N, D_model]\n")

def inspect_query_attention(query_idx: int = 7):
    query_weights = weights[query_idx].detach().numpy()
    fig = px.bar(
        x=tokens, y=query_weights,
        labels={"x": "Key Token", "y": "Attention Weight"},
        title=f"Attention weights for Query token: '{tokens[query_idx]}' (position {query_idx})"
    )
    fig.show()



### Full Attention Matrix Visualization

We can view the `[N, N]` attention weights tensor as a heatmap, showing how much every query (row) pays attention to every key (column).


In [ ]:
fig = px.imshow(
    weights.detach().numpy(),
    x=[f"{i}:{t}" for i, t in enumerate(tokens)],
    y=[f"{i}:{t}" for i, t in enumerate(tokens)],
    labels={"x": "Key Token", "y": "Query Token", "color": "Weight"},
    title="Self-Attention Heatmap [N, N]"
)
fig.update_xaxes(side="top")


## 5. Multi-Head Attention (Parallelizing Information)

Instead of a single attention operation over `D_model`, we split `D_model` into multiple "heads". 
Watch how the shape transforms dynamically using `.reshape()` and `.transpose()` to compute attention in parallel!


In [ ]:
num_heads = 4
head_dim = d_model // num_heads

print(f"[bold magenta]d_model:[/bold magenta] {d_model}")
print(f"[bold magenta]num_heads:[/bold magenta] {num_heads}")
print(f"[bold magenta]head_dim:[/bold magenta] {head_dim}  (Notice {num_heads} * {head_dim} = {d_model})\n")

# 1. Project to Q, K, V (using the same projections for simplicity)
Q_mha = q_proj(X)  # [N, d_model]
K_mha = k_proj(X)
V_mha = v_proj(X)

# 2. Reshape to split into multiple heads
# Shape goes from [N, d_model] -> [N, num_heads, head_dim]
Q_split = Q_mha.reshape(len(tokens), num_heads, head_dim)
K_split = K_mha.reshape(len(tokens), num_heads, head_dim)
V_split = V_mha.reshape(len(tokens), num_heads, head_dim)
print(f"[bold cyan]Step 2: Reshape[/bold cyan] (Splitting heads)")
print(f"Q_split shape: [bold yellow]{tuple(Q_split.shape)}[/bold yellow]  --> [N, Num_Heads, Head_Dim]\n")

# 3. Transpose so heads are the first dimension for batch matrix multiplication
# Shape goes from [N, num_heads, head_dim] -> [num_heads, N, head_dim]
Q_trans = Q_split.transpose(0, 1)
K_trans = K_split.transpose(0, 1)
V_trans = V_split.transpose(0, 1)
print(f"[bold cyan]Step 3: Transpose[/bold cyan] (Preparing for parallel attention)")
print(f"Q_trans shape: [bold yellow]{tuple(Q_trans.shape)}[/bold yellow]  --> [Num_Heads, N, Head_Dim]\n")

# 4. Compute Attention independently for each head in one operation!
# [num_heads, N, head_dim] @ [num_heads, head_dim, N] -> [num_heads, N, N]
scores_mha = Q_trans @ K_trans.transpose(-2, -1) / math.sqrt(head_dim)
weights_mha = torch.softmax(scores_mha, dim=-1)
print(f"[bold cyan]Step 4: Parallel Attention[/bold cyan]")
print(f"MHA Weights shape: [bold yellow]{tuple(weights_mha.shape)}[/bold yellow]  --> [Num_Heads, N, N]\n")

# 5. Weighted sum of Values
# [num_heads, N, N] @ [num_heads, N, head_dim] -> [num_heads, N, head_dim]
out_mha_split = weights_mha @ V_trans
print(f"[bold cyan]Step 5: Weighted Sum[/bold cyan]")
print(f"MHA Output (per head) shape: [bold yellow]{tuple(out_mha_split.shape)}[/bold yellow]  --> [Num_Heads, N, Head_Dim]\n")

# 6. Concatenate heads back together
# Transpose back to [N, num_heads, head_dim], then flatten back to [N, d_model]
out_mha_concat = out_mha_split.transpose(0, 1).reshape(len(tokens), d_model)
print(f"[bold cyan]Step 6: Concatenation[/bold cyan]")
print(f"out_mha_concat shape: [bold yellow]{tuple(out_mha_concat.shape)}[/bold yellow]  --> [N, D_model]\n")

# 7. Final linear projection
out_proj = nn.Linear(d_model, d_model)
out_mha_final = out_proj(out_mha_concat)
print(f"[bold cyan]Step 7: Final Projection[/bold cyan]")
print(f"Final MHA Output shape: [bold green]{tuple(out_mha_final.shape)}[/bold green]  --> [N, D_model]\n")

# Visualize the diverse attention patterns of the 4 heads!
query_default = 7
fig = make_subplots(rows=1, cols=num_heads, subplot_titles=[f"Head {h}" for h in range(num_heads)])
for h in range(num_heads):
    fig.add_trace(go.Bar(x=tokens, y=weights_mha[h, query_default].detach().numpy(), showlegend=False), row=1, col=h + 1)
fig.update_layout(title=f"Parallel Per-Head Attention Distributions for '{tokens[query_default]}'")


## 6. Residual Connections and Layer Normalization

We add the output of the attention block back to the original input (Residual Connection), which prevents vanishing gradients. Then we apply Layer Normalization *per token*.


In [ ]:
# Create LayerNorm layer
ln = nn.LayerNorm(d_model)

print(f"Original Input X shape:  [bold cyan]{tuple(X.shape)}[/bold cyan]")
print(f"MHA Output shape:        [bold cyan]{tuple(out_mha_final.shape)}[/bold cyan]\n")

# 1. Residual Connection (Element-wise Addition)
residual_out = X + out_mha_final
print(f"[bold magenta]1. After Residual Addition[/bold magenta] (X + MHA_out)")
print(f"shape: [bold yellow]{tuple(residual_out.shape)}[/bold yellow]\n")

# 2. Layer Normalization
normed_out = ln(residual_out)
print(f"[bold magenta]2. After LayerNorm[/bold magenta]")
print(f"shape: [bold green]{tuple(normed_out.shape)}[/bold green]\n")

# Let's see what LayerNorm did to the features of a single token
tok_idx = 7
raw_vec = residual_out[tok_idx].detach().numpy()
norm_vec = normed_out[tok_idx].detach().numpy()

# Use rich to show the pre and post layernorm stats
ln_table = Table(title=f"LayerNorm Stats for token '{tokens[tok_idx]}'", title_style="bold blue")
ln_table.add_column("Stage", style="white")
ln_table.add_column("Mean", justify="right", style="yellow")
ln_table.add_column("Std Dev", justify="right", style="cyan")

ln_table.add_row("Pre-LayerNorm", f"{raw_vec.mean():.4f}", f"{raw_vec.std():.4f}")
ln_table.add_row("Post-LayerNorm", f"{norm_vec.mean():.4f}", f"{norm_vec.std():.4f}")

console.print(ln_table)

fig = go.Figure()
fig.add_trace(go.Histogram(x=raw_vec, name="Pre-LayerNorm", opacity=0.6))
fig.add_trace(go.Histogram(x=norm_vec, name="Post-LayerNorm", opacity=0.6))
fig.update_layout(barmode="overlay", title=f"Feature distribution for token '{tokens[tok_idx]}'")


## 7. The Position-wise Feed-Forward Network (FFN)

This step operates on every token completely independently. Notice how the features are heavily expanded, passed through an activation, and compressed back down.


In [ ]:
# The FFN expands the dimension by a factor (usually 4x)
hidden_dim = 4 * d_model

fc1 = nn.Linear(d_model, hidden_dim)
act = nn.GELU()
fc2 = nn.Linear(hidden_dim, d_model)

print(f"[bold white]Input to FFN shape:[/bold white] [bold cyan]{tuple(normed_out.shape)}[/bold cyan]  --> [N, D_model]\n")

# 1. Expansion Layer + Activation
hidden_states = act(fc1(normed_out))
print(f"[bold magenta]After FC1 (Expansion) shape:[/bold magenta] [bold yellow]{tuple(hidden_states.shape)}[/bold yellow]  --> [N, Hidden_Dim]\n")

# 2. Compression Layer
ffn_out = fc2(hidden_states)
print(f"[bold green]After FC2 (Compression) shape:[/bold green] [bold cyan]{tuple(ffn_out.shape)}[/bold cyan]  --> [N, D_model]\n")

ffn_table = Table(title="FFN Expand-and-Compress Architecture", title_style="bold blue")
ffn_table.add_column("Layer", style="white")
ffn_table.add_column("Dimension", justify="right", style="cyan")
ffn_table.add_row("Input", str(d_model))
ffn_table.add_row("Hidden (Expanded)", str(hidden_dim))
ffn_table.add_row("Output (Compressed)", str(d_model))

console.print(ffn_table)

dims_df = pd.DataFrame({
    "Layer": ["Input", "Hidden (Expanded)", "Output (Compressed)"],
    "Dimension": [d_model, hidden_dim, d_model],
})
fig = px.bar(dims_df, x="Layer", y="Dimension", text="Dimension", title="FFN Expand-and-Compress Architecture")


## 8. Putting It All Together: The Complete Encoder Block

Now we wrap all our explicit tensor mathematics into clean, standard PyTorch `nn.Module` classes. 

*Note: In reality, models process text in batches. We'll simulate this by adding a batch dimension to our input sequence before passing it through the final block: `[SequenceLength, D_model]` -> `[Batch=1, SequenceLength, D_model]`.*


In [ ]:
class MultiHeadAttentionModule(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x: torch.Tensor):
        # x shape: [Batch, N, D_model]
        B, N, D = x.shape
        
        # Project and split heads: [B, N, NumHeads, HeadDim] -> [B, NumHeads, N, HeadDim]
        Q = self.q_proj(x).reshape(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).reshape(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).reshape(B, N, self.num_heads, self.head_dim).transpose(1, 2)

        # Attention calculation
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_dim)
        weights = torch.softmax(scores, dim=-1)
        
        # Weighted sum and concatenate back
        out = weights @ V # [B, NumHeads, N, HeadDim]
        out = out.transpose(1, 2).reshape(B, N, D) # Flatten heads back to [B, N, D_model]
        
        return self.out_proj(out), weights

class FeedForwardModule(nn.Module):
    def __init__(self, embed_dim: int, hidden_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, embed_dim)

    def forward(self, x: torch.Tensor):
        return self.fc2(self.act(self.fc1(x)))

class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, mlp_hidden_dim: int):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttentionModule(embed_dim=embed_dim, num_heads=num_heads)
        self.ffn = FeedForwardModule(embed_dim=embed_dim, hidden_dim=mlp_hidden_dim)

    def forward(self, x: torch.Tensor):
        # 1. Pre-Norm -> Multi-Head Attention -> Residual
        attn_out, attn_weights = self.attn(self.norm1(x))
        x = x + attn_out
        
        # 2. Pre-Norm -> Feed-Forward Network -> Residual
        x = x + self.ffn(self.norm2(x))
        
        return x, attn_weights

# Add a batch dimension to our previous sequence: [N, D_model] -> [1, N, D_model]
X_batch = X.unsqueeze(0)
print(f"[bold magenta]Final Encoder Block Input shape:[/bold magenta]  [bold cyan]{tuple(X_batch.shape)}[/bold cyan]  --> [Batch, N, D_model]\n")

# Initialize the complete block
encoder_block = TransformerEncoderBlock(embed_dim=d_model, num_heads=4, mlp_hidden_dim=4*d_model)

# Forward pass!
final_output, final_attn_weights = encoder_block(X_batch)

# Present final shapes using a Rich table
final_table = Table(title="Final Encoder Block Outputs", title_style="bold green")
final_table.add_column("Tensor", style="white")
final_table.add_column("Shape", justify="right", style="cyan")
final_table.add_column("Description", style="yellow")

final_table.add_row("Final Output", str(tuple(final_output.shape)), "[Batch, N, D_model]")
final_table.add_row("Attention Weights", str(tuple(final_attn_weights.shape)), "[Batch, NumHeads, N, N]")

console.print(final_table)

# View the final heatmap of the first head in the batch
head_idx = 0
fig = px.imshow(
    final_attn_weights[0, head_idx].detach().numpy(),
    x=[f"{i}:{t}" for i, t in enumerate(tokens)],
    y=[f"{i}:{t}" for i, t in enumerate(tokens)],
    labels={"x": "Key Token", "y": "Query Token", "color": "Weight"},
    title=f"Final Encoder Block - Attention Heatmap (Head {head_idx})"
)
fig.update_xaxes(side="top")
